# Lab 04 — Prompt Engineering for Agents

Kiel University · Agentic AI (infAgAI-01a) · Winter 2026

The lecture closed with this week's assignment: *you get a research agent whose
prompt violates most of today's principles, and your job is to diagnose its
failures, rewrite the constitution, and prove the improvement with a small
regression suite.* That is exactly what we do here — on the source-evaluation
step of our running research agent, against a local Ollama model.

**Learning objectives.** After this lab you can:

- diagnose a vague system prompt against the **four pillars** (role,
  capabilities, constraints, tool policy) and rewrite it into **operational,
  checkable** instructions,
- use **fabricated few-shot history** for format control and measure
  zero-shot vs. few-shot success rates,
- climb the **output-control ladder** — ASK → JSON mode → schema →
  validation with **bounded repair-retries** — with real parse/validate code,
- build a small **property-based regression harness** and measure contract
  pass rates across temperatures and repeated runs.

## Theory recap

Sessions 1–3 built the machinery: the agent loop (Session 02) and tools via
function calling and MCP (Session 03) define what the agent *can* do. The
prompt decides what it actually *will* do. Every loop iteration is **one big
text completion**: the model receives a single token sequence — system prompt,
tool definitions, conversation history, tool results — and predicts a
continuation. There is no hidden channel. Prompt engineering is the discipline
of deciding what is in that sequence, and where. It is the only control
surface we fully own.

### The system prompt as constitution

Chat models see a list of typed messages: **system** (operator-authored
policy, persists for the run), **user** (the task; often untrusted in
products), **assistant** (the model's own outputs, including tool-call
requests), and **tool** (results the runtime appends). A chat template
serializes the roles into special tokens in one stream — roles are a *trained
convention, not a security boundary* (Session 13 exploits exactly this).

The system prompt is the agent's **constitution**. Every durable one answers
four questions — the **four pillars**: **Role** (who the agent is, scope of
its judgement, what counts as done), **Capabilities** (what each tool is *for*,
beyond its raw schema), **Constraints** (hard rules that hold regardless of
the task, including the output contract), and **Tool policy** (budgets,
retry rules, which actions need human approval). Instructions survive long
runs only if they are **operational** — checkable ("max 300 words per
section") rather than vague ("be concise"). Vague instructions *decay*: at
step 2 the constitution is most of the context; at step 30 it is ~1%,
competing with vivid recent material.

### Format and reasoning control

**Few-shot prompting** exploits in-context learning (Brown et al., 2020): the
model continues patterns in its context, with no weight updates. In agent
code, examples are *fabricated user/assistant message pairs* — the model
cannot distinguish fabricated from real history, so one planted exchange pins
the output format better than a paragraph of prose. Dosage: 1–3 examples;
beware that models copy *content and biases* of examples, not only form.
**Chain-of-thought** (Wei et al., 2022) buys reasoning with tokens:
intermediate steps act as externalized working memory. The effect is
scale-dependent, and "Let's think step by step" alone helps (Kojima et al.,
2022). The ReAct *thought* is CoT interleaved with actions — Session 05 covers
models that internalize it.

### The output-control ladder

In an agent, output format is not cosmetic — the next loop step parses it.
Four rungs: **ASK** (request JSON in prose; a persistent few-percent failure
rate remains), **JSON MODE** (decoder guarantees valid *syntax* only),
**SCHEMA** (constrained decoding masks every token that would violate the
declared structure), **VALIDATE** (parse, check semantics, repair, bounded
retry). Failures compound: with per-step failure probability $p$, a $k$-step
run survives with probability $(1-p)^k$ — at $p = 0.03$ and $k = 30$ that is
only $\approx 40\%$. Climb as high as your API allows, and keep rung four
regardless: a schema cannot guarantee that values are *true*.

### Prompting the long run, prompt-as-code

The context has two zones: the **system zone** (byte-stable → prompt caching)
and the **appended context** (grows to 60–80% history on long runs). "Lost in
the middle" (Liu et al., 2024): long-context accuracy is U-shaped — rules
first, live task last. Finally, a prompt is **source code**: version the
triple (prompt text, model ID, parameters) and regression-test the
**contract, not the wording** — under sampling, exact-match tests are useless;
assert properties and pass rates over repeated runs. The three failure
patterns to watch: **ignored instructions**, **role drift**, **format drift**.
Every fix converts hope into a mechanism — this lab builds that machinery.

## Part A — Setup and Ollama connectivity check

We only need `ollama`, `numpy`, `pandas`, and `matplotlib` (plus optional
`ipywidgets` in the tuning part). All LLM calls go to your **local** Ollama
server — no API keys, no external network access.

In [ ]:
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ollama

MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:7b")   # any tool-capable 7-30B model works

try:
    probe = ollama.chat(model=MODEL,
                        messages=[{"role": "user",
                                   "content": "Reply with the single word: ready"}])
    print(f"Ollama is up - model '{MODEL}' replied:",
          probe["message"]["content"].strip()[:60])
except Exception as exc:
    print("Could not reach Ollama or the model is missing.")
    print("  1. Start the server :  ollama serve")
    print(f"  2. Pull the model   :  ollama pull {MODEL}")
    print(f"  (technical detail: {exc})")

In [ ]:
def chat(messages, temperature=0.7, seed=None, fmt=None):
    """One guarded LLM call. Returns the raw reply text.

    fmt=None   -> free text            (ladder rung 1: ASK)
    fmt="json" -> Ollama JSON mode     (rung 2: syntax guaranteed)
    fmt=<dict> -> JSON-schema decoding (rung 3: structure guaranteed)
    """
    options = {"temperature": temperature}
    if seed is not None:
        options["seed"] = seed
    kwargs = {"format": fmt} if fmt is not None else {}
    try:
        resp = ollama.chat(model=MODEL, messages=messages,
                           options=options, **kwargs)
    except Exception as exc:
        raise RuntimeError(
            "Ollama call failed - start the server with `ollama serve` and "
            f"pull the model with `ollama pull {MODEL}`.") from exc
    return resp["message"]["content"]

print(chat([{"role": "user", "content": "Say hello in five words or fewer."}]))

## Part B — A broken constitution

Our research agent's pipeline is *brief → search → evaluate sources → draft →
revise*. This lab isolates the **evaluate sources** step. Because labs run
offline, `data/sources.json` contains a tiny synthetic corpus of six
"fetched pages" of very different quality — a statistics office, a preprint,
an anonymous blog, a dated news piece, a vendor whitepaper, and a wiki stub.

The agent you inherit runs on the system prompt below. Read it against the
four pillars: which pillar is missing entirely? Which lines are vague intent
rather than operational instruction?

In [ ]:
DATA_PATH = os.path.join("data", "sources.json")
if not os.path.exists(DATA_PATH):                       # when run from _solutions/
    DATA_PATH = os.path.join("..", "Lab04_Prompting", "data", "sources.json")

with open(DATA_PATH, encoding="utf-8") as f:
    SOURCES = json.___(___)

df_sources = pd.DataFrame(___)
print(f"{len(SOURCES)} sources in the offline corpus")
df_sources[["id", "url", "author", "date", "title"]]

<details>
<summary><b>Click here for the solution</b></summary>

```python
DATA_PATH = os.path.join("data", "sources.json")
if not os.path.exists(DATA_PATH):                       # when run from _solutions/
    DATA_PATH = os.path.join("..", "Lab04_Prompting", "data", "sources.json")

with open(DATA_PATH, encoding="utf-8") as f:
    SOURCES = json.load(f)

df_sources = pd.DataFrame(SOURCES)
print(f"{len(SOURCES)} sources in the offline corpus")
df_sources[["id", "url", "author", "date", "title"]]
```

</details>

> **Q:** Name the four message roles relevant to agents and state who authors the content of each.
<details><summary>Click for answer</summary>

**System**: the operator/developer — persistent policy. **User**: the task and follow-ups — the end user (or text injected on their behalf). **Assistant**: the model's own prior outputs, including tool-call requests. **Tool**: results the runtime appends after executing a call. Only the assistant role is model-authored; the other three are inputs the model conditions on.

</details>

In [ ]:
WEAK_SYSTEM_PROMPT = """You are a helpful research assistant.
Be careful with sources and don't hallucinate.
Use tools when helpful, be concise, and format your output nicely."""


def format_source(source):
    """Render one corpus entry the way the agent would see a fetched page."""
    return (f"URL: {source['url']}\n"
            f"Title: {source['title']}\n"
            f"Author: {source['author'] or 'unknown'}\n"
            f"Date: {source['date'] or 'unknown'}\n"
            f"---\n{source['text']}")


TASK = ("Assess the following source for the research brief "
        "'Renewable electricity in Germany 2025'. Return a JSON object with "
        "the fields url, reliability (0..1), key_claims, flags.")

messages = [
    {"role": "___", "content": WEAK_SYSTEM_PROMPT},
    {"role": "___", "content": TASK + "\n\n" + ___(SOURCES[2])},
]
print(chat(messages, temperature=0.7, seed=101))

<details>
<summary><b>Click here for the solution</b></summary>

```python
WEAK_SYSTEM_PROMPT = """You are a helpful research assistant.
Be careful with sources and don't hallucinate.
Use tools when helpful, be concise, and format your output nicely."""


def format_source(source):
    """Render one corpus entry the way the agent would see a fetched page."""
    return (f"URL: {source['url']}\n"
            f"Title: {source['title']}\n"
            f"Author: {source['author'] or 'unknown'}\n"
            f"Date: {source['date'] or 'unknown'}\n"
            f"---\n{source['text']}")


TASK = ("Assess the following source for the research brief "
        "'Renewable electricity in Germany 2025'. Return a JSON object with "
        "the fields url, reliability (0..1), key_claims, flags.")

messages = [
    {"role": "system", "content": WEAK_SYSTEM_PROMPT},
    {"role": "user", "content": TASK + "\n\n" + format_source(SOURCES[2])},
]
print(chat(messages, temperature=0.7, seed=101))
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

The message list is what the model actually sees, serialized by the chat template into one token stream. The weak constitution goes into the `system` slot; the task plus the rendered source page form the `user` turn. We deliberately test on source 2 — the anonymous clickbait blog — because a good assessment must reject it, and the weak prompt gives the model no rule for doing so. Requesting JSON *in prose* inside the task is rung 1 of the output-control ladder: ASK.

</details>

In [ ]:
def parse_note(text):
    """Try to parse a model reply as a JSON object; return None on failure."""
    try:
        note = json.___(text)
    except json.JSONDecodeError:
        return None
    return note if isinstance(note, dict) else None


n_runs = 5
ok = 0
for i in range(n_runs):
    raw = chat(messages, temperature=0.7, seed=100 + i)
    if ___(raw) is not None:
        ok += ___
print(f"Rung 1 (ASK) with the weak prompt: {ok}/{n_runs} replies parsed as JSON")

<details>
<summary><b>Click here for the solution</b></summary>

```python
def parse_note(text):
    """Try to parse a model reply as a JSON object; return None on failure."""
    try:
        note = json.loads(text)
    except json.JSONDecodeError:
        return None
    return note if isinstance(note, dict) else None


n_runs = 5
ok = 0
for i in range(n_runs):
    raw = chat(messages, temperature=0.7, seed=100 + i)
    if parse_note(raw) is not None:
        ok += 1
print(f"Rung 1 (ASK) with the weak prompt: {ok}/{n_runs} replies parsed as JSON")
```

</details>

> **Q:** Why do vague instructions like “don't hallucinate” decay over a long agent run while sharp ones survive?
<details><summary>Click for answer</summary>

At step two the system prompt dominates the context; by step thirty it is a tiny fraction, competing against recent, vivid, concrete history. Vague instructions have no operational criterion, so local context fills the gap and behavior drifts. Checkable instructions — counts, triggers, formats — have sharp edges the model can keep matching against even when the instruction is contextually distant. Also, “don't hallucinate” gives no escape hatch; “write 'not found' when unsure” does.

</details>

## Part C — Rewriting the constitution

Now apply the editing exercise from the lecture: *take each line and ask —
could a test detect a violation of this line? If not, rewrite it until it
could, or delete it.* Complete the constitution below so that all **four
pillars** are present and every constraint is **operational**. It must
license the agent to admit ignorance, pin the output contract, and encode the
editorial rules the corpus calls for (dated sources, missing authors).

In [ ]:
STRONG_SYSTEM_PROMPT = """You are a research agent. You evaluate sources and \
write Markdown reports for a technical reader.

## Role
You assess one source at a time for the current research brief. An assessment
is finished exactly when it contains every field of the output contract below.

## Capabilities
You receive fetched pages as plain text. You do not browse: assess only what
is in front of you, never knowledge about the real site.

## Constraints
- Output exactly one JSON object - no prose, no Markdown fences.
- Fields: "url" (string, copied verbatim from the URL line of the source),
  "reliability" (float between ___ and ___),
  "key_claims" (list of short strings; only claims actually made in the text),
  "flags" (list of strings).
- Never invent URLs. If evidence for a field is missing, write "___".
- Add the flag "dated" for anything published before ___.
- Add the flag "___" if no author or organization is identifiable.
- Add the flag "promotional" if the text sells a product or service.

## Tool policy
- Exactly one assessment per source; do not re-assess or aggregate.
- Anything beyond source assessment is out of scope: refuse and say why."""

print(STRONG_SYSTEM_PROMPT)

<details>
<summary><b>Click here for the solution</b></summary>

```python
STRONG_SYSTEM_PROMPT = """You are a research agent. You evaluate sources and \
write Markdown reports for a technical reader.

## Role
You assess one source at a time for the current research brief. An assessment
is finished exactly when it contains every field of the output contract below.

## Capabilities
You receive fetched pages as plain text. You do not browse: assess only what
is in front of you, never knowledge about the real site.

## Constraints
- Output exactly one JSON object - no prose, no Markdown fences.
- Fields: "url" (string, copied verbatim from the URL line of the source),
  "reliability" (float between 0 and 1),
  "key_claims" (list of short strings; only claims actually made in the text),
  "flags" (list of strings).
- Never invent URLs. If evidence for a field is missing, write "not found".
- Add the flag "dated" for anything published before 2024.
- Add the flag "no-author" if no author or organization is identifiable.
- Add the flag "promotional" if the text sells a product or service.

## Tool policy
- Exactly one assessment per source; do not re-assess or aggregate.
- Anything beyond source assessment is out of scope: refuse and say why."""

print(STRONG_SYSTEM_PROMPT)
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

Every line passes the checkability test: the output contract names exact fields and types (a test can parse them), the reliability range is numeric (a test can compare), 'not found' is an explicit escape hatch (the practical fix for 'don't hallucinate'), and the flag rules are mechanical decision rules tied to visible metadata. Note the four pillars as section headings — role, capabilities, constraints, tool policy — exactly the structure of the lecture's research-agent constitution.

</details>

In [ ]:
strong_messages = [
    {"role": "system", "content": ___},
    {"role": "user", "content": TASK + "\n\n" + format_source(SOURCES[2])},
]

ok = 0
for i in range(n_runs):
    raw = chat(___, temperature=0.7, seed=100 + i)
    if parse_note(raw) is not None:
        ok += 1
print(f"Rung 1 (ASK) with the strong prompt: {ok}/{n_runs} replies parsed as JSON")
print("\nOne sample reply:\n", chat(strong_messages, temperature=0.7, seed=100))

<details>
<summary><b>Click here for the solution</b></summary>

```python
strong_messages = [
    {"role": "system", "content": STRONG_SYSTEM_PROMPT},
    {"role": "user", "content": TASK + "\n\n" + format_source(SOURCES[2])},
]

ok = 0
for i in range(n_runs):
    raw = chat(strong_messages, temperature=0.7, seed=100 + i)
    if parse_note(raw) is not None:
        ok += 1
print(f"Rung 1 (ASK) with the strong prompt: {ok}/{n_runs} replies parsed as JSON")
print("\nOne sample reply:\n", chat(strong_messages, temperature=0.7, seed=100))
```

</details>

> **📝 Report task R1:** Rewrite the instruction *“Be careful with
> sources.”* into an operational, checkable form for the research agent —
> at least three clauses. For **each** clause, state how an automated test
> could detect a violation.
> *No solution is provided — include your answer and a short justification in
> your lab report.*

> **Q:** The tool schemas are already sent to the API. Why should the system prompt nevertheless discuss the tools?
<details><summary>Click for answer</summary>

Schemas specify *how* to call a tool (name, parameters, types) but not *when* or *why*. Strategic knowledge — prefer `fetch_page` over snippets when quoting, search before asserting, budgets, escalation rules — is relational and mission-specific and has no place in a per-tool schema. The system prompt supplies this policy layer: that is the Capabilities pillar.

</details>

## Part D — Zero-shot vs. few-shot

Few-shot in agent code means **fabricating conversation history**: a
user/assistant pair that never happened, planted before the live task. The
model cannot distinguish fabricated from real history — from its perspective
it has already answered one of these tasks in exactly this format. Per the
lecture's dosage advice we use **one** neutral, clearly synthetic example
(remember the contamination pitfall: models copy content, not only form).

In [ ]:
EXAMPLE_SOURCE = {
    "url": "stats.example-nowhere.gov/quarterly-brief",
    "title": "Quarterly Energy Brief Q2/2025 (synthetic example)",
    "author": "National Statistics Office (synthetic example)",
    "date": "2025-07-15",
    "text": ("Wind and solar supplied 51% of electricity in Q2/2025, "
             "up 3 percentage points year on year."),
}

EXAMPLE_NOTE = {
    "url": "stats.example-nowhere.gov/quarterly-brief",
    "reliability": 0.9,
    "key_claims": ["Wind and solar supplied 51% of electricity in Q2/2025"],
    "flags": [],
}

FEW_SHOT = [
    {"role": "user", "content": TASK + "\n\n" + format_source(EXAMPLE_SOURCE)},
    {"role": "___", "content": json.dumps(___)},
]


def assess(source, system_prompt, few_shot=False, fmt=None,
           temperature=0.7, seed=None):
    """One source assessment; returns the raw reply text."""
    messages = [{"role": "system", "content": system_prompt}]
    if few_shot:
        messages += ___
    messages.append({"role": "user",
                     "content": TASK + "\n\n" + format_source(___)})
    return chat(messages, temperature=temperature, seed=seed, fmt=fmt)

<details>
<summary><b>Click here for the solution</b></summary>

```python
EXAMPLE_SOURCE = {
    "url": "stats.example-nowhere.gov/quarterly-brief",
    "title": "Quarterly Energy Brief Q2/2025 (synthetic example)",
    "author": "National Statistics Office (synthetic example)",
    "date": "2025-07-15",
    "text": ("Wind and solar supplied 51% of electricity in Q2/2025, "
             "up 3 percentage points year on year."),
}

EXAMPLE_NOTE = {
    "url": "stats.example-nowhere.gov/quarterly-brief",
    "reliability": 0.9,
    "key_claims": ["Wind and solar supplied 51% of electricity in Q2/2025"],
    "flags": [],
}

FEW_SHOT = [
    {"role": "user", "content": TASK + "\n\n" + format_source(EXAMPLE_SOURCE)},
    {"role": "assistant", "content": json.dumps(EXAMPLE_NOTE)},
]


def assess(source, system_prompt, few_shot=False, fmt=None,
           temperature=0.7, seed=None):
    """One source assessment; returns the raw reply text."""
    messages = [{"role": "system", "content": system_prompt}]
    if few_shot:
        messages += FEW_SHOT
    messages.append({"role": "user",
                     "content": TASK + "\n\n" + format_source(source)})
    return chat(messages, temperature=temperature, seed=seed, fmt=fmt)
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

The fabricated pair sits *after* the system prompt and *before* the live task — a stable prefix, which is exactly what prompt caching rewards. The assistant turn is a hand-written, perfectly formatted `SourceNote`; because the model is stateless, this planted turn reads as 'I already answered in this format once'. The example is deliberately synthetic (`example-nowhere.gov`) and topically neutral, so its content cannot leak into real assessments as a plausible citation.

</details>

In [ ]:
REQUIRED_FIELDS = {"url", "reliability", "key_claims", "flags"}


def wellformed_rate(system_prompt, few_shot, fmt=None, n=6, temperature=0.7):
    """Fraction of n runs that parse AND contain all contract fields."""
    ok = 0
    for i in range(n):
        note = parse_note(assess(SOURCES[2], system_prompt, few_shot=few_shot,
                                 fmt=fmt, temperature=temperature, seed=200 + i))
        if note is not None and REQUIRED_FIELDS <= set(___):
            ok += 1
    return ___ / ___


r_zero = wellformed_rate(STRONG_SYSTEM_PROMPT, few_shot=False)
r_few = wellformed_rate(STRONG_SYSTEM_PROMPT, few_shot=___)
print(f"zero-shot: {r_zero:.0%} well-formed")
print(f"few-shot : {r_few:.0%} well-formed")

<details>
<summary><b>Click here for the solution</b></summary>

```python
REQUIRED_FIELDS = {"url", "reliability", "key_claims", "flags"}


def wellformed_rate(system_prompt, few_shot, fmt=None, n=6, temperature=0.7):
    """Fraction of n runs that parse AND contain all contract fields."""
    ok = 0
    for i in range(n):
        note = parse_note(assess(SOURCES[2], system_prompt, few_shot=few_shot,
                                 fmt=fmt, temperature=temperature, seed=200 + i))
        if note is not None and REQUIRED_FIELDS <= set(note):
            ok += 1
    return ok / n


r_zero = wellformed_rate(STRONG_SYSTEM_PROMPT, few_shot=False)
r_few = wellformed_rate(STRONG_SYSTEM_PROMPT, few_shot=True)
print(f"zero-shot: {r_zero:.0%} well-formed")
print(f"few-shot : {r_few:.0%} well-formed")
```

</details>

> **Q:** Explain in-context learning (Brown et al., 2020). What changes inside the model when it “learns” from the few-shot examples?
<details><summary>Click for answer</summary>

Nothing changes in the weights. In-context learning means the model conditions on input-output examples placed in its context and continues the demonstrated pattern at inference time, within the forward pass. The “learning” is pattern completion over the prompt, not parameter adaptation — which is why it is instantaneous, ephemeral, and bounded by the context window.

</details>

> **Q:** Describe the example-contamination pitfall and one concrete incident pattern it produces.
<details><summary>Click for answer</summary>

The model imitates *content*, not only structure: topical or stylistic biases in the examples leak into real outputs. Concrete pattern: the agent cites a source that exists only in its few-shot block, because the fabricated assistant message presented it as a legitimate answer. Mitigation: neutral, clearly-marked, obviously synthetic examples, distinct from live data — hence our `example-nowhere.gov` URL.

</details>

## Part E — Climbing the output-control ladder

Rung 1 (ASK) you have measured. Ollama gives us the next two rungs directly:
`format="json"` is **JSON mode** (the decoder can only emit syntactically
valid JSON), and passing a **JSON schema** dict constrains decoding to your
declared structure — every token that would violate it is masked out. This is
the same machinery as function calling in Session 03, wearing a different
name. What neither rung guarantees: that the values are *true*. That is rung
4, and it stays in our code.

In [ ]:
SOURCE_NOTE_SCHEMA = {
    "type": "object",
    "properties": {
        "url": {"type": "string"},
        "reliability": {"type": "number", "minimum": ___, "maximum": ___},
        "key_claims": {"type": "array", "items": {"type": "string"}},
        "flags": {"type": "array", "items": {"type": "string"}},
    },
    "required": [___],
}

raw_json_mode = assess(SOURCES[2], STRONG_SYSTEM_PROMPT, few_shot=True,
                       fmt=___, seed=7)
raw_schema = assess(SOURCES[2], STRONG_SYSTEM_PROMPT, few_shot=True,
                    fmt=___, seed=7)
print("Rung 2, JSON mode :", raw_json_mode[:150].replace("\n", " "))
print("Rung 3, schema    :", raw_schema[:150].replace("\n", " "))

<details>
<summary><b>Click here for the solution</b></summary>

```python
SOURCE_NOTE_SCHEMA = {
    "type": "object",
    "properties": {
        "url": {"type": "string"},
        "reliability": {"type": "number", "minimum": 0, "maximum": 1},
        "key_claims": {"type": "array", "items": {"type": "string"}},
        "flags": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["url", "reliability", "key_claims", "flags"],
}

raw_json_mode = assess(SOURCES[2], STRONG_SYSTEM_PROMPT, few_shot=True,
                       fmt="json", seed=7)
raw_schema = assess(SOURCES[2], STRONG_SYSTEM_PROMPT, few_shot=True,
                    fmt=SOURCE_NOTE_SCHEMA, seed=7)
print("Rung 2, JSON mode :", raw_json_mode[:150].replace("\n", " "))
print("Rung 3, schema    :", raw_schema[:150].replace("\n", " "))
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

`format="json"` guarantees syntax only — the model may still drop the `flags` field or emit `reliability` as a string. Passing the schema dict compiles it into a grammar over tokens: at each decoding step, tokens that would violate the declared structure get probability zero. Required fields, types, and the [0, 1] range are then guaranteed *by construction*. Not guaranteed: that 0.9 is a sensible reliability for a clickbait blog — semantics stay on rung 4.

</details>

> **Q:** Describe the four rungs of the output-control ladder and the guarantee each provides.
<details><summary>Click for answer</summary>

(1) **ASK** — request JSON in prose: no guarantee, a persistent failure rate remains. (2) **JSON mode** — decoding constrained to syntactically valid JSON: syntax only. (3) **Schema enforcement** — decoding constrained to a declared JSON Schema: structure, fields, types guaranteed by token masking. (4) **Validation with bounded retry** — semantic checks in code: the only rung that addresses content correctness.

</details>

In [ ]:
def validate_note(note, source):
    """Rung 4: semantic checks no schema can express. Returns a list of errors."""
    errors = []
    if note is None:
        return ["output did not parse as a JSON object"]
    for field in ("url", "reliability", "key_claims", "flags"):
        if field not in note:
            errors.append(f"missing field: {field}")
    if "reliability" in note:
        r = note["reliability"]
        if not isinstance(r, (int, float)) or not ___ <= r <= ___:
            errors.append("reliability must be a number in [0, 1]")
    if "url" in note and note["url"] != ___:
        errors.append("___")
    if "key_claims" in note and not (isinstance(note["key_claims"], list)
                                     and len(note["key_claims"]) > 0):
        errors.append("key_claims must be a non-empty list")
    return errors


note = parse_note(assess(SOURCES[2], STRONG_SYSTEM_PROMPT, few_shot=True,
                         fmt="json", seed=42))
print("errors:", validate_note(note, SOURCES[2]) or "none - note is valid")

<details>
<summary><b>Click here for the solution</b></summary>

```python
def validate_note(note, source):
    """Rung 4: semantic checks no schema can express. Returns a list of errors."""
    errors = []
    if note is None:
        return ["output did not parse as a JSON object"]
    for field in ("url", "reliability", "key_claims", "flags"):
        if field not in note:
            errors.append(f"missing field: {field}")
    if "reliability" in note:
        r = note["reliability"]
        if not isinstance(r, (int, float)) or not 0 <= r <= 1:
            errors.append("reliability must be a number in [0, 1]")
    if "url" in note and note["url"] != source["url"]:
        errors.append("url was invented or altered - copy it verbatim")
    if "key_claims" in note and not (isinstance(note["key_claims"], list)
                                     and len(note["key_claims"]) > 0):
        errors.append("key_claims must be a non-empty list")
    return errors


note = parse_note(assess(SOURCES[2], STRONG_SYSTEM_PROMPT, few_shot=True,
                         fmt="json", seed=42))
print("errors:", validate_note(note, SOURCES[2]) or "none - note is valid")
```

</details>

In [ ]:
def assess_with_retry(source, system_prompt, max_retries=2,
                      temperature=0.7, seed=None):
    """Rung 4 with bounded repair-retry. Returns (note_or_None, attempts_used).

    On invalid output, the validation errors are fed back to the model and it
    gets another try - at most max_retries times (bounded, never infinite!).
    """
    messages = [{"role": "system", "content": system_prompt}] + FEW_SHOT
    messages.append({"role": "user",
                     "content": TASK + "\n\n" + format_source(source)})
    for attempt in range(___):
        raw = chat(messages, temperature=temperature, seed=seed, fmt="json")
        note = parse_note(raw)
        errors = validate_note(___, ___)
        if not ___:
            return ___, attempt + 1
        # repair prompt: append the failed output and the validation errors
        messages.append({"role": "___", "content": ___})
        messages.append({"role": "user",
                         "content": ("Your previous output was invalid: "
                                     + "; ".join(errors)
                                     + ". Return one corrected JSON object only.")})
    return None, max_retries + 1


note, attempts = assess_with_retry(SOURCES[4], STRONG_SYSTEM_PROMPT, seed=11)
print(f"attempts used: {attempts}")
print(json.dumps(note, indent=2) if note else "gave up after bounded retries")

> **📝 Report task R2:** Complete the cell above (`assess_with_retry`).
> *No solution is provided — include your working code in your lab report,
> plus two sentences on why the retry budget must be bounded.*

> **📝 Report task R3:** Assume a single agent step produces an
> invalid assessment with probability $p = 0.05$. Derive the probability that
> a 30-step run contains **at least one** invalid step. How does one bounded
> repair-retry (assume repair succeeds with the same independent probability
> $1-p$) change the effective per-step failure probability, and what does the
> 30-step number become? Why must validation run on **every** step rather
> than only on the final answer?
> *No solution is provided — include your derivation in your lab report.*

## Part F — Variance and a mini regression harness

The lecture's rule: assert the **contract, not the wording** — sampled runs
never match a golden reference text. We define named property checks
(invariants any correct assessment must satisfy), run each configuration $n$
times per temperature, and compare **pass rates** for the weak vs. the strong
constitution. This is a miniature of the pytest suite from the lecture, and a
preview of Session 14 (evaluation).

In [ ]:
CHECKS = {
    "parses": lambda note, src: note is not None,
    "fields": lambda note, src: (note is not None
                                 and REQUIRED_FIELDS <= set(note)),
    "range": lambda note, src: (note is not None
                                and isinstance(note.get("reliability"), (int, float))
                                and 0 <= note["reliability"] <= 1),
    "url_verbatim": lambda note, src: (note is not None
                                       and note.get("url") == src["url"]),
}


def run_suite(label, system_prompt, few_shot, fmt,
              temps=(0.0, 0.7, 1.2), n=4, source=SOURCES[2]):
    """Property-based mini regression suite: n runs per temperature."""
    rows = []
    for t in temps:
        for i in range(n):
            raw = assess(source, system_prompt, few_shot=few_shot, fmt=fmt,
                         temperature=t, seed=1000 + i)
            note = ___(raw)
            row = {"prompt": label, "temperature": t, "run": i}
            for name, check in CHECKS.items():
                row[name] = bool(___(note, source))
            rows.append(row)
    return pd.DataFrame(rows)


df_weak = run_suite("weak", WEAK_SYSTEM_PROMPT, few_shot=False, fmt=None)
df_strong = run_suite("strong", STRONG_SYSTEM_PROMPT, few_shot=True, fmt="json")
results = pd.concat([df_weak, df_strong], ignore_index=True)

check_cols = list(CHECKS)
results["all_pass"] = results[check_cols].___(axis=1)
summary = results.groupby(["prompt", "temperature"])[check_cols + ["all_pass"]].mean()
summary

<details>
<summary><b>Click here for the solution</b></summary>

```python
CHECKS = {
    "parses": lambda note, src: note is not None,
    "fields": lambda note, src: (note is not None
                                 and REQUIRED_FIELDS <= set(note)),
    "range": lambda note, src: (note is not None
                                and isinstance(note.get("reliability"), (int, float))
                                and 0 <= note["reliability"] <= 1),
    "url_verbatim": lambda note, src: (note is not None
                                       and note.get("url") == src["url"]),
}


def run_suite(label, system_prompt, few_shot, fmt,
              temps=(0.0, 0.7, 1.2), n=4, source=SOURCES[2]):
    """Property-based mini regression suite: n runs per temperature."""
    rows = []
    for t in temps:
        for i in range(n):
            raw = assess(source, system_prompt, few_shot=few_shot, fmt=fmt,
                         temperature=t, seed=1000 + i)
            note = parse_note(raw)
            row = {"prompt": label, "temperature": t, "run": i}
            for name, check in CHECKS.items():
                row[name] = bool(check(note, source))
            rows.append(row)
    return pd.DataFrame(rows)


df_weak = run_suite("weak", WEAK_SYSTEM_PROMPT, few_shot=False, fmt=None)
df_strong = run_suite("strong", STRONG_SYSTEM_PROMPT, few_shot=True, fmt="json")
results = pd.concat([df_weak, df_strong], ignore_index=True)

check_cols = list(CHECKS)
results["all_pass"] = results[check_cols].all(axis=1)
summary = results.groupby(["prompt", "temperature"])[check_cols + ["all_pass"]].mean()
summary
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

Each check is a *property*: it must hold for any correct output, regardless of wording — parseability, contract fields, numeric range, and the no-invented-URLs constitution rule. `run_suite` is the fixture loop: fixed source (our regression 'case'), fixed seeds per run index, varied temperature. The groupby-mean at the end is exactly the pass-rate view a production prompt suite would gate deployments on.

</details>

In [ ]:
pivot = results.groupby(["prompt", "temperature"])["___"].mean().unstack(0)
ax = pivot.plot(kind="bar", figsize=(7, 4), rot=0,
                color={"weak": "tab:red", "strong": "tab:green"})
ax.set_xlabel("temperature")
ax.set_ylabel("___")
ax.set_ylim(0, 1.05)
ax.set_title("Contract pass rate: weak vs. strong constitution")
ax.legend(title="prompt")
plt.tight_layout()
plt.show()

<details>
<summary><b>Click here for the solution</b></summary>

```python
pivot = results.groupby(["prompt", "temperature"])["all_pass"].mean().unstack(0)
ax = pivot.plot(kind="bar", figsize=(7, 4), rot=0,
                color={"weak": "tab:red", "strong": "tab:green"})
ax.set_xlabel("temperature")
ax.set_ylabel("pass rate (all checks)")
ax.set_ylim(0, 1.05)
ax.set_title("Contract pass rate: weak vs. strong constitution")
ax.legend(title="prompt")
plt.tight_layout()
plt.show()
```

</details>

> **📝 Report task R4:** Using your measurements from Part F: (a) explain
> why a prompt regression suite should assert a **pass rate over repeated
> runs** rather than per-run perfection, and (b) name the lecture's failure
> pattern that your data shows growing with temperature, with one fix from
> the failure-pattern table.
> *No solution is provided — include your answer, with your measured numbers,
> in your lab report.*

> **Q:** Why is exact-match (“golden output”) testing useless for agent prompts, and what replaces it?
<details><summary>Click for answer</summary>

Sampling and divergent tool trajectories mean two correct runs share almost no surface text; a golden reference would fail constantly. Replacement: **property-based testing** — assert invariants any correct output must satisfy (parses, has required sections, every claim cited, no forbidden content), and assert a pass rate over repeated runs rather than per-run perfection.

</details>

> **Q:** (not exam-relevant) Design a minimal CI setup for the research agent's prompts: tiers, mocking, and metrics.
<details><summary>Click for answer</summary>

Fast tier on every commit: fixed cases with **mocked tools** (canned search results), asserting structural and citation invariants — cheap and near-deterministic. Nightly tier: a small set with live tools to catch integration drift. Metrics: pass rate per case (e.g. require 95% over N repeats), token cost per run, latency. Block deploys on pass-rate regression; canary new prompt versions on a traffic slice with instant rollback — and re-run the whole suite on every model upgrade, because model upgrades are prompt regressions too.

</details>

## Part G — Tuning and exploration

No gaps here — this part is for experiments. Things worth trying:

- **Temperature sweep:** at which temperature does the strong configuration
  start failing checks? Does JSON mode postpone that point?
- **Ablate a pillar:** delete the `## Constraints` block from
  `STRONG_SYSTEM_PROMPT` and re-run Part F — which checks collapse first?
- **Nonlocal effects (Sclar et al., 2024):** change only *formatting* of the
  constitution (bullet style, casing, separators) and watch the pass rate.
- **Other sources:** the vendor whitepaper (index 4) and the wiki stub
  (index 5) probe the `promotional` and `no-author` rules; does the model
  apply them without being shown an example of either flag?
- **Contamination probe:** make the few-shot example topically loaded (e.g.
  strongly opinionated) and check whether its content leaks into assessments.

In [ ]:
def explore(temperature=0.7, n=4, few_shot=True, use_json_mode=True,
            source_idx=2):
    """Run the full validate pipeline n times and report the pass count."""
    fmt = "json" if use_json_mode else None
    src = SOURCES[source_idx]
    hits = 0
    for i in range(n):
        note = parse_note(assess(src, STRONG_SYSTEM_PROMPT, few_shot=few_shot,
                                 fmt=fmt, temperature=temperature,
                                 seed=5000 + i))
        if not validate_note(note, src):
            hits += 1
    print(f"T={temperature:.1f}  few_shot={few_shot}  json_mode={use_json_mode}  "
          f"source={source_idx}: {hits}/{n} runs pass every check")


try:
    import ipywidgets as widgets
    from ipywidgets import interact

    interact(explore,
             temperature=widgets.FloatSlider(min=0.0, max=1.5, step=0.1,
                                             value=0.7),
             n=widgets.IntSlider(min=2, max=10, value=4),
             few_shot=True, use_json_mode=True,
             source_idx=widgets.IntSlider(min=0, max=len(SOURCES) - 1,
                                          value=2))
except ImportError:
    print("ipywidgets not installed - calling explore() directly instead:\n")
    explore(temperature=0.2)
    explore(temperature=1.2)
    explore(temperature=1.2, use_json_mode=False)

## Wrap-up

**Takeaways.**

- The system prompt is the agent's **constitution**: role, capabilities,
  constraints, tool policy — and its language must be **operational**.
  Instructions you can check survive long runs; vibes do not (you measured
  the difference in Part F).
- **Show the format:** one fabricated few-shot exchange, or better an
  enforced schema, beats prose descriptions — and function calling and
  structured output are the same constrained-decoding mechanism.
- **Keep rung four regardless:** JSON mode and schemas guarantee syntax and
  structure, never truth. Semantic validation with **bounded** repair-retries
  is the last line of defense at every step of the loop.
- **Test the contract, not the wording:** property checks plus pass rates
  over repeated runs make prompt edits safe — prompt-as-code.

**Next week (Session 05):** reasoning models — models trained to generate
long internal chains of thought before answering. If the model plans
internally, how much of today's CoT prompting becomes redundant, and what
happens to the agent loop when single steps get slower but smarter?

---

### 📋 For your lab report

| # | Task | Where |
|---|------|-------|
| R1 | Rewrite *“Be careful with sources.”* into ≥ 3 operational clauses, each with its automated-test detector | Part C |
| R2 | Completed `assess_with_retry` code + why the retry budget must be bounded | Part E |
| R3 | Derivation: failure compounding over 30 steps with $p=0.05$, effect of one repair-retry, why per-step validation | Part E |
| R4 | Pass rate over repeated runs vs. per-run perfection; the temperature-linked failure pattern and its fix — with your measured numbers | Part F |